# Research Assist Master Worker trên Kaggle

Thêm `dist/ResearchDesk_Source.zip` vào một Kaggle Dataset riêng tư, rồi gắn dataset vào notebook. Tạo Kaggle Secret `RA_ACCESS_TOKEN` gồm 32–256 ký tự URL-safe và cấp quyền cho notebook. Bật Internet để tải thư viện hoặc trọng số. Chọn Accelerator nếu cần GPU.

Cell bên dưới kiểm tra SHA-256 của ZIP và giải nén vào một thư mục mới dưới `/kaggle/temp`.
Chọn `MODE` là `workspace`, `gpu_worker` hoặc `hybrid`. Đặt `OPEN_PUBLIC_TUNNEL = True` khi cần truy cập từ ngoài Kaggle.
Dùng token từ Secrets để đăng nhập hoặc làm Bearer token cho API quản trị và GPU.

Với GPU Embedding, dùng URL được in và gọi `GET /v1/info` bằng Bearer token để lấy model ID, tokenizer và giới hạn token.
Tạo Secrets `RA_GEMINI_API_KEY` và `RA_OPENALEX_API_KEY` nếu cần các dịch vụ này.


In [ ]:
import hashlib
import json
import os
import stat
import subprocess
import tempfile
import zipfile
from pathlib import Path, PurePosixPath

MODE = "workspace"  # workspace | gpu_worker | hybrid
OPEN_PUBLIC_TUNNEL = False
SOURCE_ZIP = ""  # Để trống để tự tìm ZIP trong /kaggle/input.

files = (
    [Path(SOURCE_ZIP)]
    if SOURCE_ZIP
    else sorted(Path("/kaggle/input").rglob("ResearchDesk_Source.zip"))
)
if len(files) != 1 or not files[0].is_file():
    raise ValueError("Gắn đúng một ResearchDesk_Source.zip hoặc đặt đường dẫn SOURCE_ZIP.")

root = Path(tempfile.mkdtemp(prefix="research-assist-master-", dir="/kaggle/temp"))
prefix = "research-assist/"
with zipfile.ZipFile(files[0]) as archive:
    items = archive.infolist()
    expected = {
        "kaggle/boot.sh",
        "kaggle/run_kaggle.py",
        "src/research_assist/cli.py",
        "pyproject.toml",
    }
    members = [entry for entry in items if not entry.is_dir()]
    if sum(entry.file_size for entry in members) > 100 * 1024**2:
        raise ValueError("ZIP v\u01b0\u1ee3t gi\u1edbi h\u1ea1n 100 MiB.")
    manifest = json.loads(archive.read(prefix + "SOURCE_MANIFEST.json"))["files"]
    names = {entry.filename for entry in members}
    if (
        len(members) != len(names)
        or names != {prefix + name for name in manifest} | {prefix + "SOURCE_MANIFEST.json"}
        or not expected.issubset(manifest)
    ):
        raise ValueError("ZIP không đúng manifest hoặc vượt giới hạn 100 MiB.")
    for name, digest in manifest.items():
        entry = archive.getinfo(prefix + name)
        path = PurePosixPath(name)
        if (
            not name
            or path.is_absolute()
            or ".." in path.parts
            or "\\" in name
            or ":" in name
            or stat.S_ISLNK(entry.external_attr >> 16)
        ):
            raise ValueError("Tệp không hợp lệ trong ZIP.")
        content = archive.read(entry)
        if hashlib.sha256(content).hexdigest() != digest:
            raise ValueError(f"SHA-256 không khớp: {name}")
        target = root.joinpath(*path.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)

command = ["bash", "kaggle/boot.sh", "--mode", MODE]
if OPEN_PUBLIC_TUNNEL:
    command.append("--public-tunnel")
runtime = {**os.environ, "PROJECT_DIR": str(root)}
subprocess.run(command, cwd=root, env=runtime, check=True)